### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="framingham_heart_study",
    dataset_year="2022",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/aasheesh200/framingham-heart-study-dataset",
    download_description="""
We download the data from Kaggle.

mkdir -p local-data-warehouse/framingham_heart_study/ && cd local-data-warehouse/framingham_heart_study && kaggle datasets download aasheesh200/framingham-heart-study-dataset && cd ../../ && unzip local-data-warehouse/framingham_heart_study/framingham-heart-study-dataset.zip -d local-data-warehouse/framingham_heart_study && rm local-data-warehouse/framingham_heart_study/framingham-heart-study-dataset.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{bhardwaj2022framingham,
	title={Framingham heart study dataset },
	url={https://www.kaggle.com/dsv/3493583},
	DOI={10.34740/KAGGLE/DSV/3493583},
	publisher={Kaggle},
	author={Ashish Bhardwaj},
	year={2022}
}
""",
    academic_reference_bibtex_key="bhardwaj2022framingham",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We mapped binary features to yes/no
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="TenYearCHD",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="TenYearCHD",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/framingham.csv", header=0)

feature_names = [
    "male",
    "age",
    "education",
    "currentSmoker",
    "cigsPerDay",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
    "TenYearCHD"
]

df.columns = feature_names

cat_features = [
    "male",
    "education",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "TenYearCHD"
]

bin_features = [
    "male",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes",
    "TenYearCHD"
]

df[cat_features] = df[cat_features].astype("category")

for col in bin_features:
    df[col] = df[col].map({0: "no", 1: "yes"})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,no,49,3.0,yes,10.0,no,no,no,no,260.0,123.0,80.0,23.10,63.0,65.0,yes
1,yes,43,1.0,yes,25.0,no,no,no,no,201.0,121.0,82.0,23.84,70.0,91.0,no
2,yes,45,1.0,yes,1.0,no,no,yes,no,277.0,140.0,84.0,28.74,69.0,74.0,no
3,no,63,3.0,yes,10.0,no,no,yes,no,236.0,189.0,103.0,27.91,60.0,74.0,no
4,yes,59,2.0,no,0.0,no,no,no,no,237.0,131.5,84.0,24.17,90.0,94.0,yes


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 4,240
Columns: 16
Use sampling: False (sample size: 4,240)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['BMI', 'totChol', 'sysBP', 'diaBP', 'glucose', 'heartRate', 'age', 'cigsPerDay', 'education', 'BPMeds']
Rows remaining as candidates after top-10 filter: 0 (of 4,240)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,no,49,3.0,yes,10.0,no,no,no,no,260.0,123.0,80.0,23.10,63.0,65.0,yes
1,yes,43,1.0,yes,25.0,no,no,no,no,201.0,121.0,82.0,23.84,70.0,91.0,no
2,yes,45,1.0,yes,1.0,no,no,yes,no,277.0,140.0,84.0,28.74,69.0,74.0,no
3,no,63,3.0,yes,10.0,no,no,yes,no,236.0,189.0,103.0,27.91,60.0,74.0,no
4,yes,59,2.0,no,0.0,no,no,no,no,237.0,131.5,84.0,24.17,90.0,94.0,yes


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,education,category,105.0,2.48,4.0,"1.0, 2.0, 3.0, 4.0"
1,BPMeds,category,53.0,1.25,2.0,"no, yes"
2,male,category,0.0,0.00,2.0,"no, yes"
3,currentSmoker,category,0.0,0.00,2.0,"no, yes"
4,prevalentStroke,category,0.0,0.00,2.0,"no, yes"
5,prevalentHyp,category,0.0,0.00,2.0,"no, yes"
6,diabetes,category,0.0,0.00,2.0,"no, yes"
7,TenYearCHD,category,0.0,0.00,2.0,"no, yes"
8,glucose,float64,388.0,9.15,143.0,"75.0, 77.0, 73.0, 80.0, 70.0, 83.0, 78.0, 74.0, 76.0, 85.0"
9,totChol,float64,50.0,1.18,248.0,"240.0, 220.0, 260.0, 210.0, 232.0, 250.0, 200.0, 230.0, 225.0, 205.0"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,4240.0,49.580189,8.572942,32.00,70.0
cigsPerDay,4211.0,9.005937,11.922462,0.00,70.0
totChol,4190.0,236.699523,44.591284,107.00,696.0
sysBP,4240.0,132.354599,22.033300,83.50,295.0
diaBP,4240.0,82.897759,11.910394,48.00,142.5
BMI,4221.0,25.800801,4.079840,15.54,56.8
heartRate,4239.0,75.878981,12.025348,44.00,143.0
glucose,3852.0,81.963655,23.954335,40.00,394.0


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                    
BPMeds          1       no   4063  95.83
                2      yes    124   2.92
                3     <NA>     53   1.25
TenYearCHD      1       no   3596  84.81
                2      yes    644  15.19
currentSmoker   1       no   2145  50.59
                2      yes   2095  49.41
diabetes        1       no   4131  97.43
                2      yes    109   2.57
education       1      1.0   1720  40.57
                2      2.0   1253  29.55
                3      3.0    689  16.25
                4      4.0    473  11.16
                5     <NA>    105   2.48
male            1       no   2420  57.08
                2      yes   1820  42.92
prevalentHyp    1       no   2923  68.94
                2      yes   1317  31.06
prevalentStroke 1       no   4215  99.41
                2      yes     25   0.59

In [9]:
# Target Distribution
target_df

,count,pct
TenYearCHD,,
no,3596,84.81
yes,644,15.19


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to framingham_heart_study/019d9cdc-9a0d-7c91-ba72-e2c31fd7c833
019d9cdc-9a0d-7c91-ba72-e2c31fd7c833
7862a7ff85073e06f63319d8255f2b7a3a74caeba127ad239e34bc8a52c50663
